

# Práctico 2.4 | Inspección y control de calidad de secuencias con R
**Bioinformática en el Diagnóstico Molecular — 2026**  
**Semana 9 · Google Colab · Lenguaje R**  
**Modalidad:** Práctico guiado

## Objetivo
Explorar un conjunto de secuenciación *paired-end*, interpretar distintas métricas de calidad y aplicar un *trimming* sencillo para comparar los datos antes y después del procesamiento.

## Ruta de trabajo
Durante el práctico iremos avanzando juntos:

1. Reconocer R1 y R2.
2. Inspeccionar la estructura FASTQ.
3. Evaluar longitud.
4. Evaluar calidad Phred por posición.
5. Comparar calidad promedio por read.
6. Explorar contenido GC.
7. Integrar la evidencia y formular un diagnóstico.
8. Aplicar *trimming*.
9. Comparar antes y después.

**Antes del procesamiento habrá una pausa obligatoria para discutir el diagnóstico.**

## Cómo usar este práctico para la tarea individual

Durante este práctico observe especialmente **cómo se construye cada bloque de código**. En la tarea recibirá la instalación y descarga de los archivos, pero deberá **adaptar por su cuenta** el código utilizado aquí para:

- importar archivos FASTQ;
- contar reads e inspeccionar su longitud;
- calcular calidad Phred por posición;
- calcular calidad promedio por read;
- calcular contenido GC;
- aplicar trimming y reevaluar.



## 1. Preparar el entorno
Usaremos **ShortRead** para FASTQ, **Biostrings** para secuencias y **ggplot2** para visualización.


In [ ]:
if (!requireNamespace("BiocManager", quietly=TRUE)) install.packages("BiocManager")
for (p in c("ShortRead","Biostrings"))
  if (!requireNamespace(p, quietly=TRUE)) BiocManager::install(p, ask=FALSE, update=FALSE)
if (!requireNamespace("ggplot2", quietly=TRUE)) install.packages("ggplot2")


In [ ]:
library(ShortRead)
library(Biostrings)
library(ggplot2)


## 2. Obtener los datos
Trabajaremos con un subconjunto de **RNA-seq paired-end** preparado para entrenamiento. Los archivos se descargarán automáticamente desde Zenodo.

- **R1:** lectura forward
- **R2:** lectura reverse


In [ ]:
url_R1 <- "https://zenodo.org/records/61771/files/GSM461178_untreat_paired_subset_1.fastq?download=1"
url_R2 <- "https://zenodo.org/records/61771/files/GSM461178_untreat_paired_subset_2.fastq?download=1"

archivo_R1 <- "muestra_R1.fastq"
archivo_R2 <- "muestra_R2.fastq"

download.file(url_R1, archivo_R1, mode="wb")
download.file(url_R2, archivo_R2, mode="wb")

file.info(c(archivo_R1, archivo_R2))[, "size", drop=FALSE]


### Pregunta 1
**¿Qué representan R1 y R2 en una secuenciación paired-end?**

## 3. ¿Qué contiene un archivo FASTQ?
Observemos las primeras dos reads de R1.



---

### ✅ Respuesta guiada 1
**R1 y R2 corresponden a las dos lecturas obtenidas desde extremos opuestos del mismo fragmento de ADN/cDNA en una secuenciación *paired-end*.**  
R1 es la lectura *forward* y R2 la lectura *reverse*. No son dos muestras distintas: forman un **par de reads del mismo fragmento**.

---


### ✅ Respuesta guiada 2
En cada registro FASTQ hay cuatro líneas:

1. `@...` → **identificador de la read**.  
2. Secuencia de letras `A`, `C`, `G`, `T` → **secuencia nucleotídica**.  
3. `+` → separador.  
4. Cadena de símbolos → **información de calidad** codificada como puntajes Phred.

Por lo tanto, **la segunda línea contiene la secuencia y la cuarta línea contiene la calidad**.


In [ ]:
readLines(archivo_R1, n=8)


Cada read ocupa cuatro líneas: **identificador → secuencia → `+` → calidad**.

### Pregunta 2
**¿Qué línea corresponde a la secuencia y cuál contiene la información de calidad?**

### Código modelo 1 — Importación e inspección
Este bloque será una referencia directa para la tarea individual.

## 4. Importar los FASTQ en R
`readFastq()` conserva conjuntamente identificadores, secuencias y valores de calidad.



---

### ✅ Respuesta guiada 2
En cada registro FASTQ hay cuatro líneas:

1. `@...` → **identificador de la read**.  
2. Secuencia de letras `A`, `C`, `G`, `T` → **secuencia nucleotídica**.  
3. `+` → separador.  
4. Cadena de símbolos → **información de calidad** codificada como puntajes Phred.

Por lo tanto, **la segunda línea contiene la secuencia y la cuarta línea contiene la calidad**.

---


In [ ]:
reads_R1 <- readFastq(archivo_R1)
reads_R2 <- readFastq(archivo_R2)

cat("Número de reads R1:", length(reads_R1), "\n")
cat("Número de reads R2:", length(reads_R2), "\n")
stopifnot(length(reads_R1) == length(reads_R2))


### Código modelo 2 — Longitud de reads
Observe qué objetos se crean y cómo se construye el gráfico.

## 5. ¿Todas las reads tienen la misma longitud?


In [ ]:
longitud_R1 <- width(sread(reads_R1))
longitud_R2 <- width(sread(reads_R2))

summary(longitud_R1)
summary(longitud_R2)

datos_longitud <- data.frame(
  longitud=c(longitud_R1,longitud_R2),
  archivo=c(rep("R1",length(longitud_R1)),rep("R2",length(longitud_R2)))
)

ggplot(datos_longitud,aes(longitud)) +
  geom_histogram(binwidth=1) +
  facet_wrap(~archivo,ncol=1) +
  labs(title="Distribución de la longitud de las reads",
       x="Longitud (pb)",y="Número de reads") +
  theme_minimal()


### Pregunta 3
**¿Las reads R1 y R2 presentan longitudes similares? ¿Qué observas en su distribución?**

### Código modelo 3 — Calidad Phred por posición
Este es uno de los bloques principales que deberá adaptar en la tarea.

## 6. ¿Cómo cambia la calidad a lo largo de las reads?
Convertiremos los puntajes **Phred** a valores numéricos y calcularemos la calidad promedio en cada posición.



---

### ✅ Respuesta guiada 3
Al ejecutar el código debemos comparar los resúmenes y los histogramas de R1 y R2. En este conjunto de entrenamiento, **R1 y R2 presentan longitudes comparables y una distribución concentrada en la longitud de lectura utilizada para la secuenciación**.

Lo importante aquí es distinguir **longitud** de **calidad**: que dos reads tengan longitudes similares no significa que tengan la misma calidad.

---


In [ ]:
Q_R1 <- as(quality(reads_R1),"matrix")
Q_R2 <- as(quality(reads_R2),"matrix")

calidad_pos_R1 <- colMeans(Q_R1,na.rm=TRUE)
calidad_pos_R2 <- colMeans(Q_R2,na.rm=TRUE)

calidad_pos <- rbind(
 data.frame(posicion=seq_along(calidad_pos_R1),calidad=calidad_pos_R1,archivo="R1"),
 data.frame(posicion=seq_along(calidad_pos_R2),calidad=calidad_pos_R2,archivo="R2")
)

ggplot(calidad_pos,aes(posicion,calidad,linetype=archivo)) +
 geom_line(linewidth=.8) +
 geom_hline(yintercept=30,linetype="dotted") +
 labs(title="Calidad promedio por posición",
      x="Posición en la read",y="Phred promedio",linetype="Read") +
 theme_minimal()


La línea Q30 es una **referencia visual**, no una regla automática de descarte.

### Pregunta 4
**¿La calidad se mantiene constante a lo largo de las reads? ¿Qué diferencias observas entre R1 y R2?**

### Código modelo 4 — Calidad promedio por read
Distinga este análisis del anterior.

## 7. ¿Todas las reads tienen la misma calidad?
Ahora evaluaremos la **calidad promedio de cada read**, en lugar de la calidad de cada posición.



---

### ✅ Respuesta guiada 4
La calidad **no tiene por qué mantenerse constante a lo largo de toda la read**. El gráfico permite observar que la calidad puede disminuir hacia las posiciones finales, es decir, hacia el **extremo 3′**.

Al comparar R1 y R2 debemos evaluarlas por separado: **R2 muestra un deterioro terminal más marcado que R1 en este ejercicio**, lo que explica que posteriormente evaluemos un recorte mayor en R2.

La línea Q30 funciona aquí como **referencia visual**, no como una regla automática para eliminar datos.

---


In [ ]:
calidad_media_R1 <- rowMeans(Q_R1,na.rm=TRUE)
calidad_media_R2 <- rowMeans(Q_R2,na.rm=TRUE)

datos_calidad <- data.frame(
 calidad=c(calidad_media_R1,calidad_media_R2),
 archivo=c(rep("R1",length(calidad_media_R1)),rep("R2",length(calidad_media_R2)))
)

ggplot(datos_calidad,aes(calidad)) +
 geom_histogram(binwidth=1) +
 facet_wrap(~archivo,ncol=1,scales="free_y") +
 labs(title="Distribución de la calidad promedio por read",
      x="Phred promedio por read",y="Número de reads") +
 theme_minimal()


### Pregunta 5
**¿Qué diferencia existe entre evaluar la calidad por posición y la calidad promedio de cada read?**

### Código modelo 5 — Contenido GC
Observe la función y cómo se aplica a R1 y R2.

## 8. ¿Qué ocurre con el contenido GC?
Una distribución GC diferente a la esperada **no significa automáticamente mala calidad**; debe interpretarse según la muestra y el protocolo.



---

### ✅ Respuesta guiada 5
Son dos preguntas diferentes:

- **Calidad por posición:** resume la calidad de todas las reads en cada posición y permite localizar **dónde** aparece un problema, por ejemplo hacia el extremo 3′.
- **Calidad promedio por read:** resume cada read completa en un único valor y permite reconocer si existen reads globalmente de mayor o menor calidad.

Por eso, una buena calidad promedio por read **puede ocultar una disminución localizada en las posiciones finales**.

---


In [ ]:
calcular_GC <- function(reads){
 f <- alphabetFrequency(sread(reads),baseOnly=TRUE)
 validas <- rowSums(f[,c("A","C","G","T"),drop=FALSE])
 100*(f[,"G"]+f[,"C"])/validas
}
gc_R1 <- calcular_GC(reads_R1)
gc_R2 <- calcular_GC(reads_R2)

datos_gc <- data.frame(
 GC=c(gc_R1,gc_R2),
 archivo=c(rep("R1",length(gc_R1)),rep("R2",length(gc_R2)))
)

ggplot(datos_gc,aes(GC)) +
 geom_histogram(binwidth=1) +
 facet_wrap(~archivo,ncol=1,scales="free_y") +
 labs(title="Distribución del contenido GC",
      x="Contenido GC (%)",y="Número de reads") +
 theme_minimal()


### Pregunta 6
**¿Cómo describirías la distribución del contenido GC? ¿Sería suficiente esta métrica por sí sola para descartar los datos? Justifica.**

# 9. Diagnóstico antes de procesar

## ⛔ Pausa de interpretación
**No continúe todavía al trimming.** Complete la tabla y discuta sus observaciones con el curso.

| Característica | R1 | R2 |
|---|---|---|
| Longitud | | |
| Calidad al inicio | | |
| Calidad hacia 3′ | | |
| Calidad promedio por read | | |
| Contenido GC | | |

### Pregunta 7
**¿Cuál es el principal problema de calidad que identifica? ¿En qué read y en qué región se concentra? Fundamente utilizando al menos dos resultados obtenidos.**

**Discusión guiada:** antes de continuar, debemos ser capaces de responder:  
**¿Qué problema observamos? → ¿Dónde está? → ¿Qué evidencia lo respalda? → ¿Sería razonable procesar?**

### Código modelo 6 — Trimming
En la tarea deberá adaptar este procedimiento después de formular su diagnóstico.

# 10. Procesamiento de las reads: trimming del extremo 3′

Una vez realizado el diagnóstico, evaluaremos el efecto de retirar una pequeña región terminal del extremo **3′**.

> Los puntos de corte utilizados a continuación son **didácticos y específicos de este práctico**. No constituyen una regla universal para datos de secuenciación.

El objetivo es comprender qué cambia cuando retiramos posiciones terminales y por qué siempre debemos volver a evaluar los datos después del procesamiento.



---

### ✅ Respuesta guiada 6
Debemos describir la **forma y posición de las distribuciones de GC de R1 y R2** y observar si son semejantes o presentan desplazamientos importantes.

**El contenido GC por sí solo no es suficiente para descartar los datos.** Una distribución particular puede reflejar características biológicas de la muestra o del protocolo. Debe interpretarse junto con la calidad Phred, longitud, presencia de adaptadores y otras métricas de control de calidad.

---


In [ ]:
# Longitud máxima observada antes del trimming
longitud_max_R1 <- max(width(sread(reads_R1)))
longitud_max_R2 <- max(width(sread(reads_R2)))

# Puntos de corte didácticos
corte_R1 <- longitud_max_R1 - 5
corte_R2 <- longitud_max_R2 - 10

cat("Longitud original máxima R1:", longitud_max_R1, "pb\n")
cat("Longitud después del trimming R1:", corte_R1, "pb\n\n")

cat("Longitud original máxima R2:", longitud_max_R2, "pb\n")
cat("Longitud después del trimming R2:", corte_R2, "pb\n")


In [ ]:
# Trimming del extremo 3′
reads_R1_trim <- narrow(reads_R1, start = 1, end = corte_R1)
reads_R2_trim <- narrow(reads_R2, start = 1, end = corte_R2)

# Comprobamos que seguimos conservando los pares
stopifnot(length(reads_R1_trim) == length(reads_R2_trim))

cat("Pares antes del trimming:", length(reads_R1), "\n")
cat("Pares después del trimming:", length(reads_R1_trim), "\n\n")

cat("Longitud R1 antes:", unique(width(sread(reads_R1))), "\n")
cat("Longitud R1 después:", unique(width(sread(reads_R1_trim))), "\n\n")

cat("Longitud R2 antes:", unique(width(sread(reads_R2))), "\n")
cat("Longitud R2 después:", unique(width(sread(reads_R2_trim))), "\n")


# 11. ¿Qué cambió después del procesamiento?

Ahora repetiremos la evaluación de calidad por posición.

La comparación **antes vs. después** permite responder dos preguntas:

1. ¿El trimming retiró la región de menor calidad?
2. ¿Qué información se perdió al acortar las reads?

La finalidad no es que todos los valores queden sobre Q30, sino comprobar si la acción aplicada fue coherente con el problema identificado.


In [ ]:
Q_R1_trim <- as(quality(reads_R1_trim), "matrix")
Q_R2_trim <- as(quality(reads_R2_trim), "matrix")

calidad_pos_R1_trim <- colMeans(Q_R1_trim, na.rm = TRUE)
calidad_pos_R2_trim <- colMeans(Q_R2_trim, na.rm = TRUE)

comparacion <- rbind(
  data.frame(
    posicion = seq_along(calidad_pos_R1),
    calidad = calidad_pos_R1,
    archivo = "R1",
    estado = "Antes"
  ),
  data.frame(
    posicion = seq_along(calidad_pos_R1_trim),
    calidad = calidad_pos_R1_trim,
    archivo = "R1",
    estado = "Después"
  ),
  data.frame(
    posicion = seq_along(calidad_pos_R2),
    calidad = calidad_pos_R2,
    archivo = "R2",
    estado = "Antes"
  ),
  data.frame(
    posicion = seq_along(calidad_pos_R2_trim),
    calidad = calidad_pos_R2_trim,
    archivo = "R2",
    estado = "Después"
  )
)

ggplot(comparacion,
       aes(x = posicion, y = calidad, linetype = estado)) +
  geom_line(linewidth = 0.8) +
  geom_hline(yintercept = 30, linetype = "dotted") +
  facet_wrap(~ archivo, ncol = 1) +
  labs(
    title = "Calidad por posición: antes y después del trimming",
    x = "Posición en la read",
    y = "Phred promedio",
    linetype = "Estado"
  ) +
  theme_minimal()


### Pregunta 8

**¿Qué cambió después del trimming?**

En tu respuesta considera:

- qué ocurrió con la región de menor calidad;
- cómo cambió la longitud de R1 y R2;
- qué información se perdió al retirar bases del extremo 3′.

**¿Consideras que el procesamiento aplicado fue coherente con el problema identificado? Justifica.**

## 12. Guardar los FASTQ procesados

Guardaremos las reads recortadas como nuevos archivos FASTQ.  
Observa que seguimos conservando R1 y R2 como un par.



---

### ✅ Respuesta guiada 8
Después del *trimming*:

- **R1 pierde 5 bases** del extremo 3′.
- **R2 pierde 10 bases** del extremo 3′.
- Se conserva el **mismo número de pares de reads**, porque aquí estamos acortando las secuencias, no eliminando pares completos.
- Se retiran posiciones terminales, precisamente donde observamos el deterioro de calidad.
- Como consecuencia, las reads quedan **más cortas** y se pierde información de secuencia del extremo 3′.

El procesamiento es coherente con el diagnóstico porque actúa sobre la región donde se observó el problema. Sin embargo, esto no significa que *trimming* siempre sea necesario ni que esos puntos de corte sean universales.

---


In [ ]:
writeFastq(
  reads_R1_trim,
  "muestra_R1_trim.fastq.gz",
  compress = TRUE
)

writeFastq(
  reads_R2_trim,
  "muestra_R2_trim.fastq.gz",
  compress = TRUE
)

file.info(c(
  "muestra_R1_trim.fastq.gz",
  "muestra_R2_trim.fastq.gz"
))[, "size", drop = FALSE]


# 13. Decisión final

Completa la siguiente tabla:

| Característica | Antes | Después |
|---|---|---|
| Número de pares de reads |  |  |
| Longitud de R1 |  |  |
| Longitud de R2 |  |  |
| Calidad hacia 3′ |  |  |
| Región de menor calidad |  |  |

### Pregunta 9

**¿Consideras que los datos procesados son adecuados para continuar con un análisis bioinformático posterior? Fundamenta tu decisión utilizando al menos dos resultados obtenidos durante el práctico.**

---

# Cierre

El control de calidad es un proceso de toma de decisiones:

### **detectar → interpretar → decidir → procesar → volver a evaluar**

**¿Qué problema detectamos? → ¿Qué hicimos? → ¿Qué cambió? → ¿Podemos continuar?**

> En este práctico utilizamos un trimming posicional sencillo para comprender la lógica del procesamiento. Herramientas especializadas como **fastp** automatizan y amplían este tipo de procedimientos.

# Cierre del práctico

Antes de pasar a la tarea individual, compruebe que puede explicar:

- ¿Qué representan R1 y R2?
- ¿Qué información contiene un FASTQ?
- ¿Qué diferencia existe entre calidad por posición y calidad promedio por read?
- ¿Por qué una sola métrica no es suficiente para evaluar una muestra?
- ¿Qué información se pierde al realizar trimming?
- ¿Por qué debemos reevaluar los datos después del procesamiento?

La tarea formativa utilizará **muestras diferentes** y deberá resolverse de manera individual.



---

### ✅ Respuesta guiada 9
La decisión debe basarse en la comparación **antes vs. después**. En este ejercicio, si al ejecutar el gráfico final observamos que el *trimming* retiró la región terminal de menor calidad y que los pares de reads se conservaron, **los datos procesados muestran una mejora coherente con el problema identificado y pueden considerarse más adecuados para continuar el flujo bioinformático**.

Evidencias mínimas para fundamentarlo:

1. La región 3′ de menor calidad fue retirada o reducida.
2. Se mantuvo el número de pares de reads.
3. La consecuencia del procesamiento fue una reducción conocida de la longitud: 5 bases en R1 y 10 bases en R2.

La conclusión no debe basarse únicamente en “superar Q30”; debe integrar varias métricas.

---
